# Hybrid Allergen Detection: MobileBERT + Rule-Based System
## Improved Version with HTML cleaning, optimized rules, per‑class thresholds, and negation handling

## Overview
Combines rule-based and ML predictions for robust allergen detection.

## Workflow
1. Load pre-trained model
2. Define rule-based patterns
3. Hybrid prediction logic
4. Evaluate on test set
5. Save configuration


In [1]:
import torch
import numpy as np
import pandas as pd
import re
import ast
from functools import lru_cache

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')

print("Imports completed.")

Imports completed.


In [2]:
# Paths
MODEL_PATH = "../models/mobilebert_allergen_final/"   # adjust to your final model
DATA_PATH = "../data/final/labeled_dataset.csv"
BEST_THRESHOLDS_PATH = "../models/best_thresholds.npy"

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
model.to(device)
model.eval()

Using device: cuda


Loading weights:   0%|          | 0/1113 [00:00<?, ?it/s]

MobileBertForSequenceClassification(
  (mobilebert): MobileBertModel(
    (embeddings): MobileBertEmbeddings(
      (word_embeddings): Embedding(30522, 128, padding_idx=0)
      (position_embeddings): Embedding(512, 512)
      (token_type_embeddings): Embedding(2, 512)
      (embedding_transformation): Linear(in_features=384, out_features=512, bias=True)
      (LayerNorm): NoNorm()
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): MobileBertEncoder(
      (layer): ModuleList(
        (0-23): 24 x MobileBertLayer(
          (attention): MobileBertAttention(
            (self): MobileBertSelfAttention(
              (query): Linear(in_features=128, out_features=128, bias=True)
              (key): Linear(in_features=128, out_features=128, bias=True)
              (value): Linear(in_features=512, out_features=128, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): MobileBertSelfOutput(
              (dense): Linear(in_fe

In [4]:
# Allergen classes
ALLERGENS = ["milk", "eggs", "peanuts", "tree_nuts", "soy", "wheat", "fish", "shellfish"]
num_labels = len(ALLERGENS)

## 1. Load and clean data

In [5]:
df = pd.read_csv(DATA_PATH)

# Clean HTML tags from ingredients
def clean_html(text):
    return re.sub(r'<[^>]+>', '', str(text))

df["clean_text"] = df["ingredients_cleaned"].apply(clean_html)

# Parse labels
def encode_labels(label_str):
    if pd.isna(label_str) or label_str == "[]":
        return [0] * len(ALLERGENS)
    labels = ast.literal_eval(label_str)
    return [1 if allergen in labels else 0 for allergen in ALLERGENS]

df["labels"] = df["detected_allergens"].apply(encode_labels)

print(f"Dataset shape: {df.shape}")
print(f"Sample cleaned text: {df['clean_text'].iloc[0][:200]}")

Dataset shape: (1057, 17)
Sample cleaned text: water, refined sugar, citric acid, artificial orange flavor,xanthan gum, ascorbic acid, sodium benzoate, sweetener


In [6]:
test_indices = np.load("../data/final/test_indices.npy")
test_df = df.iloc[test_indices]
test_texts = test_df["clean_text"].tolist()
test_labels = test_df["labels"].tolist()

In [7]:
# Load saved indices
train_indices = np.load("../data/final/train_indices.npy")
val_indices   = np.load("../data/final/val_indices.npy")
test_indices  = np.load("../data/final/test_indices.npy")

# Create the dataframes (or just extract texts and labels)
train_texts = df.iloc[train_indices]["clean_text"].tolist()
train_labels = df.iloc[train_indices]["labels"].tolist()

val_texts   = df.iloc[val_indices]["clean_text"].tolist()
val_labels  = df.iloc[val_indices]["labels"].tolist()

test_texts  = df.iloc[test_indices]["clean_text"].tolist()
test_labels = df.iloc[test_indices]["labels"].tolist()

print(f"Train size: {len(train_texts)} (positive samples: {(np.array(train_labels).sum(axis=1)>0).sum()})")
print(f"Val size:   {len(val_texts)}   (positive samples: {(np.array(val_labels).sum(axis=1)>0).sum()})")
print(f"Test size:  {len(test_texts)}  (positive samples: {(np.array(test_labels).sum(axis=1)>0).sum()})")

Train size: 739 (positive samples: 533)
Val size:   159   (positive samples: 118)
Test size:  159  (positive samples: 110)


## 2. Rule‑Based System – Optimised with pre‑compiled patterns and negation handling

In [8]:
# Expanded keyword lists with more variants
ALLERGEN_RULES = {
    "milk": [
        "milk", "skim milk", "whole milk", "milk powder", "nonfat milk", "evaporated milk", "milk solids",
        "whey", "whey protein", "casein", "caseinate",
        "butter", "butterfat", "buttermilk", "cream",
        "cheese", "cheddar", "mozzarella",
        "lactose", "lactalbumin", "lactoglobulin",
        "ghee", "custard", "yogurt"
    ],
    "eggs": [
        "egg", "eggs", "egg white", "egg yolk",
        "albumin", "ovalbumin", "lysozyme", "globulin", "ovomucoid",
        "mayonnaise", "meringue"
    ],
    "peanuts": [
        "peanut", "peanuts", "groundnut",
        "peanut butter", "peanut oil"
    ],
    "tree_nuts": [
        "almond", "cashew", "walnut", "pecan", "hazelnut",
        "macadamia", "pistachio", "brazil nut", "chestnut",
        "pine nut", "marzipan", "praline",
        "nut paste", "nut butter"
    ],
    "soy": [
        "soy", "soya", "soybean", "soybeans", "edamame", "natto",
        "soy lecithin", "lecithin",
        "soy protein", "textured vegetable protein",
        "tvp", "tofu", "miso", "tempeh"
    ],
    "wheat": [
        "wheat", "whole wheat", "wheat flour", "spelt", "kamut", "triticale", "durum",
        "flour", "gluten", "semolina",
        "farina", "bran", "bulgur",
        "bread", "breadcrumbs", "pasta", "noodles"
    ],
    "fish": [
        "fish", "tuna", "salmon", "sardine",
        "anchovy", "mackerel", "cod",
        "fish sauce", "fish oil"
    ],
    "shellfish": [
        "shrimp", "prawn", "crab", "lobster",
        "crayfish", "krill", "clam", "mussel", "oyster", "scallop", "crawfish", "langoustine",
        "shellfish extract"
    ]
}

# Pre‑compile regex patterns for each allergen
import re
compiled_rules = {}
for allergen, keywords in ALLERGEN_RULES.items():
    pattern = r'\b(?:' + '|'.join(re.escape(kw) for kw in keywords) + r')\b'
    compiled_rules[allergen] = re.compile(pattern, re.IGNORECASE)

# Negation pattern (simple, can be expanded)
NEGATION_PATTERN = re.compile(r'\b(no|free|without|minus|low\s+in|none)\s+\w*\b', re.IGNORECASE)

def rule_match(text, allergen):
    # Returns True if allergen keyword found and not negated.
    
    text_lower = text.lower()
    if compiled_rules[allergen].search(text_lower):
        if NEGATION_PATTERN.search(text_lower):
            return False
        return True
    return False

## 3. ML Prediction Function

In [9]:
def predict_ml(texts, thresholds=0.5, max_length=209):
    if isinstance(texts, str):
        texts = [texts]
    inputs = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
    probs = torch.sigmoid(outputs.logits).cpu().numpy()
    preds = (probs >= thresholds).astype(int)
    return preds, probs

In [10]:
# After you have train_texts, compute max_len
sample_lengths = [len(tokenizer.encode(t, truncation=False)) for t in train_texts[:500]]
max_len = int(np.percentile(sample_lengths, 95))
MAX_LEN = min(max_len, 256)
print(f"Using max_length = {MAX_LEN}")

Using max_length = 209


## 4. Hybrid Predictor – with flexible rule integration

In [11]:
def hybrid_predict(texts, ml_thresholds=None, rule_conf_threshold=0.2, mode='hard_override', alpha=0.3):
    # mode: 'hard_override' (if rule matches and ML prob > rule_conf_threshold, set to 1),
    #       'soft' (final_prob = ml_prob + alpha * rule_match, then threshold 0.5),
    #       'high_confidence_bypass' (if ML prob > 0.9, ignore rule).

    ml_preds, probs = predict_ml(texts, thresholds=ml_thresholds)
    hybrid_preds = []
    for i, text in enumerate(texts):
        final_preds = ml_preds[i].copy()
        for j, allergen in enumerate(ALLERGENS):
            rule_present = rule_match(text, allergen)
            if mode == 'hard_override':
                if rule_present and probs[i][j] > rule_conf_threshold:
                    final_preds[j] = 1
            elif mode == 'soft':
                final_prob = probs[i][j] + alpha * (1 if rule_present else 0)
                final_preds[j] = 1 if final_prob > 0.5 else 0
            elif mode == 'high_confidence_bypass':
                if rule_present and probs[i][j] > rule_conf_threshold and probs[i][j] < 0.9:
                    final_preds[j] = 1
        hybrid_preds.append(final_preds)
    return np.array(hybrid_preds), probs

## 5. Load optimal ML thresholds (pre‑computed from validation)

In [12]:
from scipy.special import expit
from sklearn.metrics import f1_score

def find_best_thresholds(probs, labels, step=0.01):
    num_classes = probs.shape[1]
    best_thresholds = []
    for i in range(num_classes):
        best_t = 0.5
        best_f1 = 0
        for t in np.arange(0.01, 0.99, step):
            preds = (probs[:, i] >= t).astype(int)
            f1 = f1_score(labels[:, i], preds, zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                best_t = t
        best_thresholds.append(best_t)
    return np.array(best_thresholds)

# Get validation predictions
val_ml_preds, val_probs = predict_ml(val_texts, max_length=MAX_LEN)
best_thresholds = find_best_thresholds(val_probs, np.array(val_labels))
print("Computed thresholds:", best_thresholds)

Computed thresholds: [0.04 0.04 0.01 0.05 0.02 0.43 0.92 0.1 ]


## 6. Tune rule confidence threshold (global) on validation set

In [13]:
best_global_rule_threshold = 0.2
best_global_f1 = 0
print("Tuning global rule confidence threshold on validation set...")
for conf in np.arange(0.05, 0.55, 0.05):
    val_hybrid_preds, _ = hybrid_predict(val_texts, ml_thresholds=best_thresholds,
                                         rule_conf_threshold=conf, mode='hard_override')
    macro_f1 = f1_score(val_labels, val_hybrid_preds, average='macro', zero_division=0)
    print(f"conf={conf:.2f} -> macro F1={macro_f1:.4f}")
    if macro_f1 > best_global_f1:
        best_global_f1 = macro_f1
        best_global_rule_threshold = conf
print(f"\nBest global rule threshold: {best_global_rule_threshold:.2f} (macro F1={best_global_f1:.4f})")

Tuning global rule confidence threshold on validation set...
conf=0.05 -> macro F1=0.9736
conf=0.10 -> macro F1=0.9736
conf=0.15 -> macro F1=0.9736
conf=0.20 -> macro F1=0.9736
conf=0.25 -> macro F1=0.9736
conf=0.30 -> macro F1=0.9736
conf=0.35 -> macro F1=0.9736
conf=0.40 -> macro F1=0.9736
conf=0.45 -> macro F1=0.9736
conf=0.50 -> macro F1=0.9736

Best global rule threshold: 0.05 (macro F1=0.9736)


## 7. Evaluate on Test Set

In [14]:
# ML only baseline
ml_test_preds, _ = predict_ml(test_texts, thresholds=best_thresholds)
print("=== ML ONLY (test set) ===")
print(classification_report(test_labels, ml_test_preds, target_names=ALLERGENS, zero_division=0))
print(f"Macro F1: {f1_score(test_labels, ml_test_preds, average='macro', zero_division=0):.4f}\n")

# Hybrid (hard override with global optimal threshold)
test_hybrid_preds, _ = hybrid_predict(test_texts, ml_thresholds=best_thresholds,
                                      rule_conf_threshold=best_global_rule_threshold,
                                      mode='hard_override')
print("=== HYBRID (hard override) ===")
print(classification_report(test_labels, test_hybrid_preds, target_names=ALLERGENS, zero_division=0))
print(f"Macro F1: {f1_score(test_labels, test_hybrid_preds, average='macro', zero_division=0):.4f}")

# Try alternative mode: high confidence bypass
test_hybrid_bypass, _ = hybrid_predict(test_texts, ml_thresholds=best_thresholds,
                                       rule_conf_threshold=best_global_rule_threshold,
                                       mode='high_confidence_bypass')
print("\n=== HYBRID (high confidence bypass) ===")
print(classification_report(test_labels, test_hybrid_bypass, target_names=ALLERGENS, zero_division=0))
print(f"Macro F1: {f1_score(test_labels, test_hybrid_bypass, average='macro', zero_division=0):.4f}")

=== ML ONLY (test set) ===
              precision    recall  f1-score   support

        milk       0.97      0.92      0.95        76
        eggs       1.00      0.90      0.95        10
     peanuts       0.80      1.00      0.89         4
   tree_nuts       0.91      0.96      0.93        45
         soy       0.93      1.00      0.96        52
       wheat       1.00      0.98      0.99        59
        fish       1.00      1.00      1.00        12
   shellfish       1.00      0.60      0.75         5

   micro avg       0.96      0.95      0.96       263
   macro avg       0.95      0.92      0.93       263
weighted avg       0.96      0.95      0.96       263
 samples avg       0.65      0.65      0.65       263

Macro F1: 0.9277

=== HYBRID (hard override) ===
              precision    recall  f1-score   support

        milk       0.97      0.92      0.95        76
        eggs       1.00      0.90      0.95        10
     peanuts       0.80      1.00      0.89         4
  

## 8. Error Analysis (examples where hybrid differs from ML)

In [15]:
results_df = pd.DataFrame({
    "text": test_texts,
    "true": [list(l) for l in test_labels],
    "ml_pred": [list(p) for p in ml_test_preds],
    "hybrid_pred": [list(p) for p in test_hybrid_preds]
})
differences = results_df[results_df["ml_pred"] != results_df["hybrid_pred"]]
print(f"Number of test samples where hybrid changed prediction: {len(differences)}")
if len(differences) > 0:
    print("\nFirst 3 examples:")
    for i in range(min(3, len(differences))):
        row = differences.iloc[i]
        true_all = [ALLERGENS[j] for j, v in enumerate(row['true']) if v == 1]
        ml_all = [ALLERGENS[j] for j, v in enumerate(row['ml_pred']) if v == 1]
        hybrid_all = [ALLERGENS[j] for j, v in enumerate(row['hybrid_pred']) if v == 1]
        print(f"\nText: {row['text'][:150]}...")
        print(f"True: {true_all}")
        print(f"ML:   {ml_all}")
        print(f"Hybrid:{hybrid_all}")

Number of test samples where hybrid changed prediction: 0


## 9. Save hybrid configuration for production

In [16]:
import json
hybrid_config = {
    "ml_thresholds": best_thresholds.tolist(),
    "rule_conf_threshold": best_global_rule_threshold,
    "mode": "hard_override"
}
with open("../models/hybrid_config.json", "w") as f:
    json.dump(hybrid_config, f, indent=2)
print("Hybrid configuration saved to ../models/hybrid_config.json")

Hybrid configuration saved to ../models/hybrid_config.json
